# Export the categoriser for the public web prototype

Produces `models/ledger_web_model.json`, the model that runs inside the browser on ledger.danishali.dev/try/.
It is the same TF-IDF + logistic regression as `expense_categoriser_train_test.ipynb`, with two changes:

1. **Privacy pruning.** The model file is public, so its vocabulary must not reveal anyone's merchants.
   A character or word n-gram is kept only if it occurs in the expenses of at least `MIN_USERS` different people.
   Anything that only one person ever bought from (a landlord, a small business, a PayPal handle) is dropped.
2. **One calibrated model instead of five.** `CalibratedClassifierCV(..., ensemble=False)`: sigmoid calibration fitted on
   grouped (unseen-merchant) out-of-fold scores, applied to one model trained on all data. The review threshold is
   re-chosen with the same rule as the main notebook (lowest threshold with ≥ 90% accuracy on auto-filed unseen merchants).

The merchant lookup table is **not** exported (it lists everyone's merchants). In the browser, tier 1 is the visitor's own
corrections instead. Nothing from the Khalifa dataset is used for training, only for the external score.

Run all cells. It takes a few minutes on the CPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Masters of Applied AI/MAAI902/sms_expense_project_bundle"

In [ ]:
import json, re, datetime, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, accuracy_score
import sklearn
warnings.filterwarnings('ignore', category=UserWarning)

TRANSACTIONS_DIR, LABELS_FILE, MODEL_DIR = Path('outputs'), Path('merchant_labels.xlsx'), Path('models')
MIN_USERS = 2                 # keep an n-gram only if at least this many people's expenses contain it
TARGET_AUTO_ACCURACY = 0.90   # same rule as the main notebook
print('scikit-learn', sklearn.__version__)

In [ ]:
# Same loading, labels and features as expense_categoriser_train_test.ipynb (sections 2-3)
personal = pd.read_csv(TRANSACTIONS_DIR / 'personal_transactions_expenses.csv', parse_dates=['date'])
khalifa = pd.read_csv(TRANSACTIONS_DIR / 'khalifa_transactions_expenses.csv', parse_dates=['date'])
merchants = pd.read_excel(LABELS_FILE, sheet_name='Merchants')
CATS = [c for c in pd.read_excel(LABELS_FILE, sheet_name='Categories').category if c != 'Total']
label_map = dict(zip(merchants.merchant, merchants.final_category))
ov = pd.read_excel(LABELS_FILE, sheet_name='Ambiguous rows')
overrides = {k: v for k, v in zip(ov.txn_id, ov.row_override) if isinstance(v, str)}

def attach_labels(df):
    df = df.copy()
    df['label'] = df.merchant_clean.map(label_map)
    has = df.txn_id.isin(overrides)
    df.loc[has, 'label'] = df.loc[has, 'txn_id'].map(overrides)
    return df

def make_text(df):
    t = df.merchant.fillna('').astype(str).str.upper()
    t = t.str.replace(r'\+?\d[\d -]{4,}', ' ', regex=True)
    t = t.str.replace(r'[^A-Z& ]', ' ', regex=True)
    return t.str.replace(r'\s+', ' ', regex=True).str.strip()

personal, khalifa = attach_labels(personal), attach_labels(khalifa)
for d in (personal, khalifa):
    assert d.label.notna().all()
    d['text'] = make_text(d)
    d['log_amount'] = np.log1p(d.amount_aed.fillna(d.amount).clip(lower=0))
FEATURES = ['text', 'log_amount']
print(len(personal), 'personal expenses from', personal.user.nunique(), 'people;', len(khalifa), 'Khalifa')

In [ ]:
# 1. Privacy pruning: which n-grams occur in at least MIN_USERS people's expenses?
CHAR = dict(analyzer='char_wb', ngram_range=(2, 5))
WORD = dict(analyzer='word', ngram_range=(1, 2))

def shared_vocab(params):
    cv = CountVectorizer(binary=True, **params).fit(personal.text)
    X = cv.transform(personal.text)
    users = personal.user.values
    present = np.vstack([np.asarray(X[users == u].max(axis=0).todense()).ravel() for u in np.unique(users)])
    n_users = present.sum(axis=0)
    terms = np.array(cv.get_feature_names_out())
    return sorted(terms[n_users >= MIN_USERS]), len(terms)

char_vocab, char_all = shared_vocab(CHAR)
word_vocab, word_all = shared_vocab(WORD)
print(f'char n-grams kept {len(char_vocab):,} of {char_all:,}; words kept {len(word_vocab):,} of {word_all:,}')

In [ ]:
# 2. Same model, with and without pruning
def tfidf_logreg(char_vocab=None, word_vocab=None):
    text = FeatureUnion([
        ('char', TfidfVectorizer(sublinear_tf=True, min_df=1, vocabulary=char_vocab, **CHAR)),
        ('word', TfidfVectorizer(sublinear_tf=True, min_df=1, vocabulary=word_vocab, **WORD)),
    ])
    pre = ColumnTransformer([('text', text, 'text'), ('amount', StandardScaler(), ['log_amount'])])
    return Pipeline([('features', pre), ('clf', LogisticRegression(max_iter=3000, C=10, class_weight='balanced'))])

pruned = lambda: tfidf_logreg(char_vocab, word_vocab)
full = lambda: tfidf_logreg()

def unseen_merchant_scores(factory):
    pred = np.empty(len(personal), dtype=object)
    for tr, te in GroupKFold(n_splits=5).split(personal, groups=personal.merchant_clean):
        m = factory().fit(personal.iloc[tr][FEATURES], personal.iloc[tr].label)
        pred[te] = m.predict(personal.iloc[te][FEATURES])
    return f1_score(personal.label, pred, average='macro', labels=CATS, zero_division=0), accuracy_score(personal.label, pred)

def external_scores(factory):
    m = factory().fit(personal[FEATURES], personal.label)
    p = m.predict(khalifa[FEATURES])
    return f1_score(khalifa.label, p, average='macro', labels=CATS, zero_division=0), accuracy_score(khalifa.label, p)

# Note: the main notebook averages macro-F1 over the 5 folds; this pools them, so it can differ by a point.
scores = pd.DataFrame({
    'full model (report)': [*unseen_merchant_scores(full), *external_scores(full)],
    f'pruned (n-grams from >= {MIN_USERS} people)': [*unseen_merchant_scores(pruned), *external_scores(pruned)],
}, index=['unseen merchants macro-F1', 'unseen merchants accuracy', 'Khalifa macro-F1', 'Khalifa accuracy'])
scores.style.format('{:.1%}')

In [ ]:
# 3. Calibrated export model + review threshold (unseen-merchant worst case, as in section 10 of the main notebook)
def calibrated(train):
    folds = list(GroupKFold(n_splits=5).split(train, groups=train.merchant_clean))
    return CalibratedClassifierCV(pruned(), method='sigmoid', cv=folds, ensemble=False).fit(train[FEATURES], train.label)

def in_cat_order(m, X):
    p, cls = m.predict_proba(X), list(m.classes_)
    return np.column_stack([p[:, cls.index(c)] if c in cls else np.zeros(len(p)) for c in CATS])

oof = np.zeros((len(personal), len(CATS)))
for tr, te in GroupKFold(n_splits=5).split(personal, groups=personal.merchant_clean):
    oof[te] = in_cat_order(calibrated(personal.iloc[tr]), personal.iloc[te][FEATURES])
conf, right = oof.max(1), np.array(CATS)[oof.argmax(1)] == personal.label.values

rows = []
for t in np.round(np.arange(0.3, 0.96, 0.05), 2):
    auto = conf >= t
    rows.append(dict(threshold=float(t), auto_filed=float(auto.mean()),
                     accuracy_on_auto=float(right[auto].mean()) if auto.any() else None))
thr = pd.DataFrame(rows)
ok = thr[thr.accuracy_on_auto >= TARGET_AUTO_ACCURACY]
REVIEW_THRESHOLD = float(ok.threshold.min()) if len(ok) else 0.9
bins = pd.cut(conf, [0, .3, .4, .5, .6, .7, .8, .9, 1.0])
rel = pd.DataFrame({'bin': bins.astype(str), 'conf': conf, 'right': right}).groupby('bin').agg(
    n=('right', 'size'), mean_confidence=('conf', 'mean'), actual_accuracy=('right', 'mean'))
print('REVIEW_THRESHOLD =', REVIEW_THRESHOLD)
display(thr.style.format({'auto_filed': '{:.0%}', 'accuracy_on_auto': '{:.0%}'}))
rel.round(2)

In [ ]:
# 4. Fit on all personal data and write the JSON
cal = calibrated(personal)
assert len(cal.calibrated_classifiers_) == 1
cc = cal.calibrated_classifiers_[0]
base = cc.estimator
pre, clf = base.named_steps['features'], base.named_steps['clf']
parts = dict(pre.named_transformers_['text'].transformer_list)

def vec_dict(v):
    vocab = sorted(v.vocabulary_, key=v.vocabulary_.get)
    return dict(analyzer=v.analyzer, ngram=list(v.ngram_range), vocab=vocab, idf=[round(float(x), 6) for x in v.idf_])

# Self-check strings (made up, not from anyone's data): the browser test compares its scores with these.
CHECK = ['TALABAT', 'KARAK HOUSE JLT', 'AL MAYA SUPERMARKET', 'DUBAI POLICE FINES', 'NETFLIX COM', 'ASTER PHARMACY',
         'CAREEM FOOD', 'SHELL', 'CINEMA CITY AL ZAHIA', 'FLYDUBAI', 'ICP SMART SERVICES', 'GEMS WELLINGTON SCHOOL',
         'XYZ HOLDINGS LLC', 'AL NOOR TRADING EST', 'CARREFOUR MOE', 'ADNOC', 'MCDONALDS ADNOC', 'IKEA', 'DEWA', '']
CHECK_AMT = [40, 12, 230, 300, 56, 80, 45, 120, 70, 1850, 300, 15000, 500, 80, 230, 150, 30, 900, 450, 50]
cx = pd.DataFrame({'text': CHECK, 'log_amount': np.log1p(CHECK_AMT)})
check_p = in_cat_order(cal, cx).round(6).tolist()

model = dict(
    categories=CATS, classes=[str(c) for c in clf.classes_],
    char=vec_dict(parts['char']), word=vec_dict(parts['word']),
    amount=dict(mean=float(pre.named_transformers_['amount'].mean_[0]), scale=float(pre.named_transformers_['amount'].scale_[0])),
    coef=[[round(float(x), 5) for x in row] for row in clf.coef_], intercept=[round(float(x), 6) for x in clf.intercept_],
    calibration=dict(method='sigmoid', a=[float(c.a_) for c in cc.calibrators], b=[float(c.b_) for c in cc.calibrators]),
    review_threshold=REVIEW_THRESHOLD,
    check=dict(text=CHECK, amount=CHECK_AMT, proba=check_p),
    meta=dict(created=datetime.datetime.now().isoformat(timespec='minutes'), sklearn=sklearn.__version__,
              min_users=MIN_USERS, n_train=int(len(personal)), people=int(personal.user.nunique()),
              scores={k: {i: round(float(v), 4) for i, v in col.items()} for k, col in scores.items()},
              threshold_table=[{k: (None if isinstance(v, float) and v != v else v) for k, v in r.items()} for r in thr.to_dict('records')],
              note='Privacy-pruned, calibrated export of the MAAI902 expense categoriser. No lookup table, no transactions.'),
)
assert list(cc.classes) == model['classes']
assert clf.coef_.shape[1] == len(model['char']['vocab']) + len(model['word']['vocab']) + 1

MODEL_DIR.mkdir(exist_ok=True)
path = MODEL_DIR / 'ledger_web_model.json'
path.write_text(json.dumps(model, separators=(',', ':'), allow_nan=False))   # browsers reject NaN
print(f'Saved {path} ({path.stat().st_size/1e6:.1f} MB), threshold {REVIEW_THRESHOLD}')

**Privacy check before publishing:** every n-gram in the file occurs in at least two people's expenses, the merchant
lookup table is not included, and no transaction, amount, date or name is stored. The `check` block contains only the
made-up strings above.